In [1]:
# Cell 1: Install dependencies (uncomment if you haven't installed geemap yet)
!pip install earthengine-api geemap geedim

  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached affine-3.0.1-py3-none-any.whl.metadata (4.5 kB)
  Using cached cligj-0.7.2-py3-none-any.whl.metadata (5.0 kB)
  Using cached click_plugins-1.1.1.2-py2.py3-none-any.whl.metadata (6.5 kB)
Using cached attrs-26.1.0-py3-none-any.whl (67 kB)
Using cached fsspec-2026.9.0-py3-none-any.whl (221 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 4.1 MB/s  0:00:05m0:00:0100:01m
Using cached cligj-0.7.2-py3-none-any.whl (7.1 kB)
Using cached tabulate-0.10.0-py3-none-any.whl (39 kB)
Using cached tqdm-4.70.1-py3-none-any.whl (80 kB)
Using cached affine-3.0.1-py3-none-any.whl (10 kB)
Using cached click_plugins-1.1.1.2-py2.py3-none-any.whl (11 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [geedim]14/16 [aiohttp]]gins]


In [1]:
import ee
import geemap

# -------------------------------------------------------------------------
# 1. Initialize Google Earth Engine
# -------------------------------------------------------------------------
try:
    ee.Initialize(project='XX')  # Replace 'your-project-id' with your actual GCP project ID
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='XX')  # Replace 'your-project-id' with your actual GCP project ID

print("Earth Engine successfully initialized.")

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine successfully initialized.


In [4]:
# -------------------------------------------------------------------------
# 2. Define AOI & Filter Sentinel-1 Collection
# -------------------------------------------------------------------------
# Hawke's Bay Coastal & River Region (Esk Valley / Napier)
aoi = ee.Geometry.BBox(176.60, -39.60, 177.10, -39.00)

s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(aoi)
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
      .filter(ee.Filter.eq('instrumentMode', 'IW'))
      .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')))

# -------------------------------------------------------------------------
# 3. Select Pre- and Post-Event Scene Pair (Track 1)
# -------------------------------------------------------------------------
pre_image = s1.filterDate('2023-02-01', '2023-02-03').first().clip(aoi)
post_image = s1.filterDate('2023-02-13', '2023-02-15').first().clip(aoi)

# Verify Scene Acquisition Timestamps
print("Pre-Event Acquisition Date :", pre_image.date().format('YYYY-MM-dd HH:mm:ss z').getInfo())
print("Post-Event Acquisition Date:", post_image.date().format('YYYY-MM-dd HH:mm:ss z').getInfo())

Pre-Event Acquisition Date : 2023-02-01 17:30:20 UTC
Post-Event Acquisition Date: 2023-02-13 17:30:31 UTC


In [8]:
# -------------------------------------------------------------------------
# 4. Speckle Reduction & Difference Surface
# -------------------------------------------------------------------------
smoothing_radius = 30  # meters
pre_vv = pre_image.select('VV').focalMean(smoothing_radius, 'circle', 'meters')
post_vv = post_image.select('VV').focalMean(smoothing_radius, 'circle', 'meters')
pre_vh = pre_image.select('VH').focalMean(smoothing_radius, 'circle', 'meters')
post_vh = post_image.select('VH').focalMean(smoothing_radius, 'circle', 'meters')

# Calculate log-ratio difference (dB)
vv_diff = post_vv.subtract(pre_vv).rename('vv_diff')
vh_diff = post_vh.subtract(pre_vh).rename('vh_diff')

# -------------------------------------------------------------------------
# 5. Permanent Water Masking & Flood Baseline Thresholding
# -------------------------------------------------------------------------
jrc_water = ee.Image('JRC/GSW1_4/GlobalSurfaceWater')
permanent_water = jrc_water.select('occurrence').gt(80)
land_area_mask = permanent_water.Not()

flood_threshold = -3.5  # dB drop threshold
flood_mask = vv_diff.lt(flood_threshold).And(land_area_mask).rename('baseline_mask')

# -------------------------------------------------------------------------
# 6. Combine Stack & Cast Uniformly to Float32
# -------------------------------------------------------------------------
export_stack = ee.Image.cat([
    pre_vv.rename('pre_vv'),
    pre_vh.rename('pre_vh'),
    post_vv.rename('post_vv'),
    post_vh.rename('post_vh'),
    vv_diff.rename('vv_diff'),
    flood_mask.rename('baseline_mask').float()  # Convert byte mask to float32
]).toFloat()

# -------------------------------------------------------------------------
# 7. Direct Local Download (Bypasses Google Drive)
# -------------------------------------------------------------------------
output_filename = 'CycloneGabrielle_HawkesBay_S1_Stack.tif'

print("\nStarting direct local download...")
geemap.download_ee_image(
    export_stack,
    filename=output_filename,
    region=aoi,
    scale=10,
    crs='EPSG:32760',  # UTM zone 60S, so 10 m pixels are true metres
)

import os
if os.path.exists(output_filename):
    print(f"Success! Saved '{output_filename}'.")
else:
    print("Download failed — no file written.")



Starting direct local download...


/usr/local/lib/python3.13/dist-packages/geemap/common.py:12333: FutureWarning: 'BaseImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  img = gd.download.BaseImage(image)


  0%|          |0/252 tiles [00:00<?]

/usr/local/lib/python3.13/dist-packages/geedim/image.py:254: RuntimeWarning: Couldn't find STAC entry for: 'None'.
  return STACClient().get(self.id)


Success! Saved 'CycloneGabrielle_HawkesBay_S1_Stack.tif'.
